# Demo — Steps 1 & 2 (thin harness)

This notebook contains **no logic**: it only imports from `src/` and runs it. Real implementation lives in `src/` and is covered by `uv run pytest`.

> Requires IB Gateway/TWS running (paper), API enabled on the configured port.

In [ ]:
# Jupyter runs an asyncio loop already -> ib_insync needs startLoop() BEFORE connecting.
from ib_insync import util
util.startLoop()
import sys, pathlib, datetime as dt
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
from utils.config import load_config
cfg = load_config()
cfg.ibkr, cfg.environment.name

## Step 1 — Connectivity & health

In [ ]:
from connectivity.session import IBSession
session = IBSession(cfg.ibkr).connect()
print('state:', session.state.value)
session.health_check(cfg.qc.clock_skew_tolerance_sec)

### Bootstrap evidence (no orders placed)

In [ ]:
from connectivity.diagnostics import gather_evidence
gather_evidence(session, cfg)

## Step 2 — Instrument master

In [ ]:
from universe.master import resolve_underlying, get_option_chain
spy, raw = resolve_underlying(session, 'SPY')
spy

In [ ]:
chain = get_option_chain(session, spy)
print('expiries:', len(chain.expirations), '| strikes:', len(chain.strikes), '| mult:', chain.multiplier)
sorted(chain.expirations)[:5]

### Build + persist the active universe (maturity window = 7 days)

In [ ]:
from universe.master import load_active_universe
from universe.store import UniverseStore, config_fingerprint
store = UniverseStore(cfg.environment.data_dir)
summary = load_active_universe(session, ['SPY'], dt.date.today(), store=store,
                               config_fp=config_fingerprint({'src': cfg.universe.source}),
                               maturity_days=7)
summary

In [ ]:
df = store.load_options(dt.date.today().isoformat())
print('rows:', len(df)); df.head()

### Clean disconnect

In [ ]:
session.disconnect(); session.state.value

---
## Rigorous tests
```bash
uv run pytest -q
```